<a href="https://colab.research.google.com/github/janosv246/MPA-MLF-Project/blob/main/MPA_MLF_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MPA-MLF Project**  

Vít Janoš, Kryštof Havrda

### 0. Import libraries

In [1]:

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from keras.optimizers import Adam


#### 1.1 Set paths

In [ ]:
# Change these paths only if your files are somewhere else in Colab
TRAIN_DIR = '/x_train'
TEST_DIR = '/x_test'
TRAIN_CSV = '/y_train_v2.csv'
SAMPLE_SUBMISSION_CSV = '/y_test_submission_example_v2.csv'

#### 1.2 Load labels and image paths

In [ ]:

def numeric_file_key(path_obj: Path):
    stem = path_obj.stem
    digits = ''.join(ch for ch in stem if ch.isdigit())
    return int(digits) if digits else stem


def find_id_and_label_columns(df: pd.DataFrame):
    lower_cols = [str(c).strip().lower() for c in df.columns]

    id_candidates = ['id', 'image', 'image_id', 'img_id', 'index']
    label_candidates = ['label', 'labels', 'class', 'target', 'y']

    id_col = None
    label_col = None

    for original, lower in zip(df.columns, lower_cols):
        if lower in id_candidates and id_col is None:
            id_col = original
        if lower in label_candidates and label_col is None:
            label_col = original

    if df.shape[1] == 1:
        label_col = df.columns[0]
        id_col = None
    elif df.shape[1] >= 2:
        if id_col is None:
            id_col = df.columns[0]
        if label_col is None:
            label_col = df.columns[1]

    return id_col, label_col


train_df = pd.read_csv(TRAIN_CSV)
train_id_col, train_label_col = find_id_and_label_columns(train_df)

train_image_paths = sorted(Path(TRAIN_DIR).glob('*.png'), key=numeric_file_key)
print(f'Number of training images: {len(train_image_paths)}')
print('Train CSV columns:', list(train_df.columns))

if train_id_col is None:
    train_labels_df = pd.DataFrame({
        'id': np.arange(1, len(train_df) + 1),
        'label': train_df[train_label_col].to_numpy()
    })
else:
    train_labels_df = train_df[[train_id_col, train_label_col]].copy()
    train_labels_df.columns = ['id', 'label']

train_labels_df = train_labels_df.sort_values('id').reset_index(drop=True)
train_image_ids = np.array([numeric_file_key(p) for p in train_image_paths])

print(train_labels_df.head())


#### 1.3 Show a few images

In [ ]:

def load_grayscale_image(image_path, target_size=(64, 64)):
    image = Image.open(image_path).convert('L')
    image = image.resize(target_size)
    image = np.array(image, dtype=np.float32) / 255.0
    return image


def display_random_images(image_paths, labels_df, count=8, target_size=(64, 64)):
    selected_idx = np.random.choice(len(image_paths), size=count, replace=False)

    plt.figure(figsize=(16, 6))
    for i, idx in enumerate(selected_idx, 1):
        img_path = image_paths[idx]
        img_id = numeric_file_key(img_path)
        label_row = labels_df.loc[labels_df['id'] == img_id, 'label']
        label = int(label_row.iloc[0]) if len(label_row) > 0 else -1

        img = load_grayscale_image(img_path, target_size=target_size)

        plt.subplot(2, 4, i)
        plt.imshow(img, cmap='gray')
        plt.title(f'id={img_id}, class={label}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


display_random_images(train_image_paths, train_labels_df)


#### 1.4 Load and preprocess the dataset

In [ ]:

def build_dataset(image_paths, labels_df=None, target_size=(64, 64)):
    X = []
    y = []
    ids = []

    label_map = None
    if labels_df is not None:
        label_map = dict(zip(labels_df['id'].astype(int), labels_df['label'].astype(int)))

    for img_path in image_paths:
        img_id = int(numeric_file_key(img_path))
        img = load_grayscale_image(img_path, target_size=target_size)
        X.append(img)
        ids.append(img_id)

        if label_map is not None:
            y.append(label_map[img_id])

    X = np.array(X, dtype=np.float32)
    X = X.reshape(-1, target_size[0], target_size[1], 1)
    ids = np.array(ids)

    if label_map is not None:
        y = np.array(y, dtype=np.int32)
        return X, y, ids

    return X, ids


IMG_SIZE = (64, 64)
NUM_CLASSES = 4

X, y, train_ids = build_dataset(train_image_paths, train_labels_df, target_size=IMG_SIZE)
y_cat = to_categorical(y, NUM_CLASSES)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'X_train shape: {X_train.shape}')
print(f'X_val shape: {X_val.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_val shape: {y_val.shape}')
print('Classes:', np.unique(y))
